# F01 POLUX — Oracle & Curation
## PENTERACT DORN V3 — VIIe Légion

**Rôle** : Valide le `plan_de_vol.json` produit par META_POLUX et strip les EXIF de tous les PNG.

**Entrées** : `F01_POLUX/IN/plan_de_vol.json` + `F01_POLUX/IN/images/*.png`  
**Sorties** : `F01_POLUX/OUT/plan_de_vol.json` + `F01_POLUX/OUT/images/*.png`

---
### Avant de lancer
1. Monte ton Google Drive (cellule ci-dessous)
2. **Première fois uniquement** : lance CELLULE 2 — Init DRIVE_DORN (crée toute la structure + télécharge les scripts)
3. Dépose `plan_de_vol.json` dans `DRIVE_DORN/F01_POLUX/IN/`
4. Dépose tes PNG dans `DRIVE_DORN/F01_POLUX/IN/images/`
5. Lance les cellules 3 à 7 dans l'ordre

In [ ]:
# CELLULE 1 — Montage Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive monté.')

In [ ]:
# CELLULE 2 — Init DRIVE_DORN (run une seule fois — idempotente)
# Crée toute la structure de dossiers sur Drive ET télécharge les scripts depuis GitHub.
# Sûr à relancer : exist_ok=True, les fichiers existants ne sont pas écrasés.

import os, urllib.request

DRIVE_BASE = '/content/drive/MyDrive/DRIVE_DORN'
GITHUB_RAW  = 'https://raw.githubusercontent.com/kioka8877-ux/DORN/main'
GITHUB_TOKEN = ''  # ← laisser vide si le repo est public, sinon coller ton token

# ── 1. Dossiers à créer ───────────────────────────────────────────────────────
DIRS = [
    'F01_POLUX/IN/images',
    'F01_POLUX/OUT/images',
    'F01_POLUX/CODEBASE',
    'F02_CASTELLAN/IN/images',
    'F02_CASTELLAN/OUT',
    'F02_CASTELLAN/CODEBASE',
    'F03_SIGISMUND/IN/images',
    'F03_SIGISMUND/OUT',
    'F03_SIGISMUND/CODEBASE',
    'F04_INWIT/IN',
    'F04_INWIT/OUT',
    'F04_INWIT/CODEBASE',
    'SHARED/IN/images',
    'SHARED/OUT',
]

print('── Création des dossiers ──')
for d in DIRS:
    path = os.path.join(DRIVE_BASE, d)
    os.makedirs(path, exist_ok=True)
    print(f'  ✓ {path}')

# ── 2. Scripts Python à télécharger depuis GitHub ────────────────────────────
SCRIPTS = [
    ('CRS_CUSTOS.py',                          'CRS_CUSTOS.py'),
    ('F01_POLUX/CODEBASE/drn_f01_polux.py',    'F01_POLUX/CODEBASE/drn_f01_polux.py'),
    ('F02_CASTELLAN/CODEBASE/drn_f02_castellan.py', 'F02_CASTELLAN/CODEBASE/drn_f02_castellan.py'),
    ('F03_SIGISMUND/CODEBASE/drn_f03_sigismund.py', 'F03_SIGISMUND/CODEBASE/drn_f03_sigismund.py'),
    ('F04_INWIT/CODEBASE/drn_f04a_inwit.py',   'F04_INWIT/CODEBASE/drn_f04a_inwit.py'),
    ('F04_INWIT/CODEBASE/drn_f04b_inwit.py',   'F04_INWIT/CODEBASE/drn_f04b_inwit.py'),
]

print('\n── Téléchargement des scripts ──')
headers = {'Authorization': f'token {GITHUB_TOKEN}'} if GITHUB_TOKEN else {}
for github_path, local_rel in SCRIPTS:
    dst = os.path.join(DRIVE_BASE, local_rel)
    url = f'{GITHUB_RAW}/{github_path}'
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req) as resp, open(dst, 'wb') as f:
            f.write(resp.read())
        print(f'  ✓ {local_rel}')
    except Exception as e:
        print(f'  ✗ {local_rel} — {e}')

print('\n══ INIT DRIVE_DORN TERMINÉ ══')
print(f'Structure créée sous : {DRIVE_BASE}')
print('Prochaine étape : dépose plan_de_vol.json dans F01_POLUX/IN/ puis lance CELLULE 3.')

In [ ]:
# CELLULE 3 — Configuration
DRIVE_BASE = '/content/drive/MyDrive/DRIVE_DORN'  # ← adapter si besoin
print(f'Drive base : {DRIVE_BASE}')

In [ ]:
# CELLULE 4 — Installation Pillow (déjà présent sur Colab, mais au cas où)
import importlib
if importlib.util.find_spec('PIL') is None:
    import subprocess
    subprocess.run(['pip', 'install', 'Pillow', '-q'], check=True)
    print('Pillow installé.')
else:
    print('Pillow déjà disponible.')

In [ ]:
# CELLULE 5 — Copie du script depuis le repo Drive
import shutil, os
script_src = os.path.join(DRIVE_BASE, 'F01_POLUX', 'CODEBASE', 'drn_f01_polux.py')
script_dst = '/content/drn_f01_polux.py'
shutil.copy2(script_src, script_dst)
print(f'Script copié : {script_dst}')

In [ ]:
# CELLULE 6 — Exécution F01 POLUX
import subprocess, sys
result = subprocess.run(
    [sys.executable, '/content/drn_f01_polux.py', '--drive-base', DRIVE_BASE],
    capture_output=False
)
if result.returncode == 0:
    print('\n✓ F01 POLUX — VALIDATION OK')
    print('→ Étape suivante : META_CAMERA (Gemini) puis F02 CASTELLAN')
else:
    print('\n✗ F01 POLUX — VALIDATION FAIL — corriger les erreurs ci-dessus')

In [ ]:
# CELLULE 7 — Transit CRS_CUSTOS (check-in F01)
# À lancer UNIQUEMENT si la cellule 6 s'est terminée avec VALIDATION OK
custos_src = os.path.join(DRIVE_BASE, 'CRS_CUSTOS.py')
shutil.copy2(custos_src, '/content/CRS_CUSTOS.py')
result2 = subprocess.run(
    [sys.executable, '/content/CRS_CUSTOS.py',
     '--frigate', 'F01', '--mode', 'check-in', '--drive-base', DRIVE_BASE],
    capture_output=False
)
if result2.returncode == 0:
    print('\n✓ CRS_CUSTOS — F01 check-in OK — Transit autorisé vers F02')
else:
    print('\n✗ CRS_CUSTOS — F01 check-in FAIL')